In [ ]:
# Run this cell first — fixes table and equation alignment throughout the notebook
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 1 Lab — The Data Science Workflow
**Introduction to Data Science — DBAS 5015**

---

<!-- ============================================================
  WRITING STYLE — apply to all markdown cells in this notebook
  - Direct and to the point. Say what the concept is and move on.
  - No hedging. State things plainly.
  - Active voice. 'The function returns a value' not 'A value is returned.'
  - Short sentences for instructions. Students need to scan them quickly.
  - Exercise instructions must be unambiguous — one reading, one meaning.
  - No filler. Every sentence should add information or cut it.
  ============================================================ -->

### What This Lab Covers
- Verifying your Python environment and imports
- numpy arrays, element-wise operations, and boolean indexing
- pandas DataFrames — loading, inspecting, selecting, and aggregating
- First-look workflow for a new dataset
- *(Optional)* End-to-end ML pipeline preview

**Estimated time:** 60–90 minutes (+ 20 min for the optional section)

---

## Part 1 — Environment Check

Run the cell below. If it executes without errors, your environment is ready. If any import fails, install the missing library before continuing (`pip install <library_name>` in a terminal).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import sklearn

print("numpy:",     np.__version__)
print("pandas:",    pd.__version__)
print("sklearn:",   sklearn.__version__)
print("\nAll imports successful.")

---
## Part 2 — numpy Arrays

A numpy array is a fixed-type collection of numbers. Operations apply to every element at once — no loops needed.

| Operation | Syntax | Result |
|:----------|:-------|:-------|
| Create from list | `np.array([1, 2, 3])` | Array |
| Element-wise math | `arr * 1.10` | New array, each value multiplied |
| Summary statistics | `np.mean(arr)`, `np.std(arr)` | Scalar |
| Boolean filter | `arr[arr > 100]` | Array of values matching condition |
| Count matching | `(arr > 100).sum()` | Integer |

Run the cell below to see these in action.

In [ ]:
# Monthly revenue for a 12-month period ($000s)
revenue = np.array([142, 138, 164, 159, 172, 181, 168, 175, 190, 185, 201, 215])

# Summary statistics
print(f"Mean:    ${np.mean(revenue):.1f}k")
print(f"Median:  ${np.median(revenue):.1f}k")
print(f"Std dev: ${np.std(revenue, ddof=1):.1f}k")
print(f"Min:     ${np.min(revenue)}k  |  Max: ${np.max(revenue)}k")

# Element-wise: project 8% growth
projected = revenue * 1.08
print(f"\nProjected (8% growth): {np.round(projected, 1)}")

# Boolean indexing: months above $170k
above_170 = revenue[revenue > 170]
print(f"\nMonths above $170k: {above_170}")
print(f"Count: {(revenue > 170).sum()} of {len(revenue)} months")

---
### 🤖 ML Connection — Why numpy Matters

scikit-learn passes data as numpy arrays internally. When you hand a pandas DataFrame to a model, it converts it. Understanding boolean indexing now means you will recognize what is happening when scikit-learn filters training samples, applies masks, or returns prediction arrays. The pattern is always the same — a condition produces a True/False array, and that array selects elements.

---

### ✏️ Exercise 2.1 — Regional Sales Analysis

You have quarterly sales figures (in $000s) for four regions. Using only numpy:

1. Calculate the mean and standard deviation for each region.
2. Find the overall company mean across all 16 values combined.
3. Identify which individual quarterly values exceed $200k.
4. Calculate what each region's Q4 value would be after a 12% year-end bonus is applied.

In [ ]:
# Exercise 2.1 — provided data, do not change
northeast = np.array([185, 192, 178, 210])
west      = np.array([162, 171, 168, 195])
south     = np.array([143, 138, 155, 172])
midwest   = np.array([121, 129, 133, 148])

# 1. Mean and std for each region
# your code here


# 2. Overall company mean (hint: combine arrays with np.concatenate)
all_regions = 
company_mean = 
print(f"Company mean: ${company_mean:.1f}k")

# 3. Values exceeding $200k across all regions
over_200 = 
print(f"Values over $200k: {over_200}")

# 4. Q4 values (index 3) with 12% bonus applied
# your code here


---
## Part 3 — pandas First-Look Workflow

Every time you open a new dataset, run these five commands in order before doing anything else:

| Command | What It Tells You |
|:--------|:------------------|
| `df.shape` | Row and column count |
| `df.dtypes` | Data type of each column |
| `df.head()` | First 5 rows — spot-check the values |
| `df.info()` | Types + non-null counts — reveals missing data immediately |
| `df.describe()` | Summary stats for numeric columns |

The cell below creates a sample sales dataset. Work through the five commands on it.

In [ ]:
# Sample sales dataset — 200 transactions
import numpy as np
import pandas as pd

np.random.seed(42)
n = 200

regions   = np.random.choice(['Northeast', 'West', 'South', 'Midwest'], n)
products  = np.random.choice(['Laptop Stand', 'Monitor Arm', 'Keyboard Tray', 'Cable Manager'], n)
units     = np.random.randint(1, 20, n)
unit_price = np.random.choice([49.95, 79.95, 34.95, 24.95], n)
revenue   = np.round(units * unit_price * np.random.uniform(0.85, 1.15, n), 2)
discount  = np.random.choice([0, 0.05, 0.10, 0.15, 0.20], n)

# Introduce some missing values
revenue_with_nulls = revenue.astype(float)
null_idx = np.random.choice(n, 12, replace=False)
revenue_with_nulls[null_idx] = np.nan

df = pd.DataFrame({
    'transaction_id': range(1001, 1001 + n),
    'region':         regions,
    'product':        products,
    'units_sold':     units,
    'unit_price':     unit_price,
    'discount_pct':   discount,
    'revenue':        revenue_with_nulls,
})

print("Dataset created:", df.shape)

In [ ]:
# Step 1: Shape
print("Shape:", df.shape)
print()

# Step 2: Data types
print("Dtypes:")
print(df.dtypes)
print()

In [ ]:
# Step 3: First 5 rows
df.head()

In [ ]:
# Step 4: Info — look at the Non-Null Count column
df.info()

In [ ]:
# Step 5: Summary statistics
df.describe()

---
### 💼 Business Context — What the First Look Tells You

`df.info()` shows 188 non-null values for `revenue` out of 200 rows — 12 missing. That is 6% of the dataset. Before you model anything, you need a decision: drop those rows, fill them with a reasonable value, or flag them for investigation. Missing data is not a technical nuisance — it is a business question. A missing revenue value might mean a transaction was voided, a system error occurred, or the data was never collected. The right handling depends on which it is.

---

### ✏️ Exercise 3.1 — Interrogate the Dataset

Using the `df` created above:

1. Select only the `region`, `product`, and `revenue` columns.
2. Filter to rows where `discount_pct` is greater than 0 (discounted transactions only).
3. Calculate the mean revenue by region using `groupby`.
4. Add a calculated column called `net_revenue` equal to `revenue * (1 - discount_pct)`.
5. How many transactions have missing revenue? Write the code to find out — do not use the number you observed from `df.info()`.

In [ ]:
# Exercise 3.1

# 1. Select three columns
subset = 
print(subset.head())

# 2. Discounted transactions only
discounted = 
print(f"\nDiscounted transactions: {len(discounted)}")

# 3. Mean revenue by region
regional_mean = 
print("\nMean revenue by region:")
print(regional_mean)

# 4. Net revenue column
df['net_revenue'] = 
print("\nnet_revenue column added:", 'net_revenue' in df.columns)

# 5. Count of missing revenue values
missing_count = 
print(f"\nMissing revenue values: {missing_count}")

---
## Part 4 — Selecting and Aggregating

Two pandas patterns you will use in every project:

**`.loc[]` for conditional row selection with specific columns:**
```python
df.loc[condition, ['col1', 'col2']]
```

**`.agg()` for multiple summary statistics at once:**
```python
df.groupby('category')['value'].agg(count='count', mean='mean', std='std')
```

In [ ]:
# Full summary table by region
region_summary = df.groupby('region')['revenue'].agg(
    count  = 'count',
    mean   = 'mean',
    median = 'median',
    std    = 'std',
    total  = 'sum'
).round(2)

print(region_summary)
print()

# High-value Northeast transactions
high_ne = df.loc[
    (df['region'] == 'Northeast') & (df['revenue'] > 500),
    ['transaction_id', 'product', 'units_sold', 'revenue']
]
print(f"High-value Northeast transactions: {len(high_ne)}")
print(high_ne.head())

### ✏️ Exercise 4.1 — Product Performance Summary

1. Build a summary table grouped by `product` showing count, mean, and total revenue. Sort by total revenue descending.
2. Use `.loc[]` to select all transactions where `product` is `'Monitor Arm'` and `units_sold` is greater than 10. Print the result.
3. Which product has the highest mean revenue per transaction? Write one sentence interpreting what this means for the business.

In [ ]:
# Exercise 4.1

# 1. Product summary table, sorted by total revenue
product_summary = 
print(product_summary)

# 2. Monitor Arm transactions with more than 10 units
monitor_large = 
print(f"\nMonitor Arm large orders: {len(monitor_large)}")
print(monitor_large)

# 3. One-sentence interpretation (write as a comment)
# 

---
## Part 5 — Challenge Exercise

This section is optional — attempt it if you finish early or want a stretch.

You receive a second dataset: 6 months of customer support tickets, with the following columns: `ticket_id`, `customer_id`, `region`, `product`, `issue_type` (`billing`, `technical`, `shipping`), `resolution_days`, and `resolved` (True/False).

Create this dataset yourself using `np.random.seed(99)` and the patterns from Part 3, then:

1. Run the full five-step first-look workflow.
2. Find the mean resolution time by `issue_type`.
3. Identify the region with the highest proportion of unresolved tickets.
4. Merge the support dataset with `df` on `region` and calculate whether high-revenue regions also have high support volumes.
5. **Reflection:** You observe that the Northeast has the highest revenue and the highest support ticket volume. A colleague concludes that selling more causes more support issues. What alternative explanation would you investigate before agreeing?

In [ ]:
# Challenge — create the support dataset, then complete tasks 1–4
np.random.seed(99)
n_tickets = 150

# Build the support DataFrame here
support = pd.DataFrame({
    'ticket_id':       range(5001, 5001 + n_tickets),
    'customer_id':     np.random.randint(1000, 2000, n_tickets),
    'region':          np.random.choice(['Northeast', 'West', 'South', 'Midwest'], n_tickets),
    'product':         np.random.choice(['Laptop Stand', 'Monitor Arm', 'Keyboard Tray', 'Cable Manager'], n_tickets),
    'issue_type':      np.random.choice(['billing', 'technical', 'shipping'], n_tickets),
    'resolution_days': np.random.randint(1, 21, n_tickets),
    'resolved':        np.random.choice([True, False], n_tickets, p=[0.82, 0.18]),
})

# Your analysis below


---
---
## ⭐ Optional — End-to-End Pipeline Preview

**This section is a preview, not a lesson.** You are not expected to understand every line. Run it, read the comments, and let it give you a picture of where this course is going.

The code below loads a dataset, prepares it, trains a model, and evaluates it — the complete data science workflow in about 30 lines. Everything here gets taught properly in Weeks 2–7. For now, just observe the shape of the process.

Three things to notice:
1. The data preparation steps outnumber the modelling steps — that ratio is realistic.
2. The scikit-learn model follows the same `.fit()` / `.predict()` / `.score()` pattern described in Chapter 1.
3. A `Pipeline` object chains preprocessing and modelling into a single step — this is what Week 7 teaches in depth.

In [ ]:
# ── End-to-End Pipeline Preview ──────────────────────────────────────────────
# Goal: predict whether a transaction's revenue is above the median (high/low)
# based on region, product, units sold, unit price, and discount.

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ── 1. Prepare the data ───────────────────────────────────────────────────────
# Drop rows with missing revenue (the 12 nulls we noticed earlier)
df_clean = df.dropna(subset=['revenue']).copy()

# Target: 1 if revenue is above median, 0 if not
median_rev = df_clean['revenue'].median()
df_clean['high_revenue'] = (df_clean['revenue'] > median_rev).astype(int)

# Features and target
features = ['region', 'product', 'units_sold', 'unit_price', 'discount_pct']
X = df_clean[features]
y = df_clean['high_revenue']

# ── 2. Split into training and test sets ──────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

# ── 3. Define preprocessing ───────────────────────────────────────────────────
# Categorical columns: encode as numbers (OneHotEncoder)
# Numeric columns: scale to a common range (StandardScaler)
categorical = ['region', 'product']
numerical   = ['units_sold', 'unit_price', 'discount_pct']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('num', StandardScaler(), numerical),
])

# ── 4. Build the pipeline ─────────────────────────────────────────────────────
# One object that preprocesses and then models — in one step
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42)),
])

# ── 5. Train and evaluate ─────────────────────────────────────────────────────
pipeline.fit(X_train, y_train)
print(f"\nTest accuracy: {pipeline.score(X_test, y_test):.2%}")
print()
print(classification_report(y_test, pipeline.predict(X_test),
                             target_names=['Low Revenue', 'High Revenue']))

### Reflection

Look at the output and answer these in a comment cell below:

1. What does the accuracy score tell you? What doesn't it tell you?
2. The pipeline has two steps: `preprocessor` and `classifier`. In plain language, what does each step do?
3. We split data into training and test sets. Why would it be a problem to evaluate the model on the same data it was trained on?

In [ ]:
# Your reflections here (as comments)

# 1. Accuracy:

# 2. What each step does:

# 3. Why train/test split matters:


---
## Week 1 Summary

| Concept | Key Takeaway |
|:--------|:-------------|
| Data science workflow | Frame → acquire → clean → model → evaluate/communicate |
| Problem framing | Define the decision, success criteria, and target variable before touching data |
| numpy array | Fixed-type numerical collection — element-wise operations, no loops |
| Boolean indexing | Condition produces True/False mask — use it to filter arrays and DataFrames |
| First-look workflow | `shape` → `dtypes` → `head()` → `info()` → `describe()` — always in this order |
| `df.info()` | Best single command for spotting missing data and type problems |
| `.groupby().agg()` | Split by category, apply multiple statistics, combine into a table |
| scikit-learn API | `.fit()` / `.predict()` / `.score()` — same pattern for every model |
| Pipeline | Chains preprocessing and modelling into one object — prevents data leakage |

---
**Next week:** pandas in depth — handling missing data, fixing data types, and identifying the features that matter for analysis and modelling.